# 8-Queens Problem as a CSP: Backtracking vs MRV+LCV vs AC-3

This notebook implements the 8-Queens problem as a **Constraint Satisfaction Problem (CSP)** and solves it using three approaches:

1. **Plain Backtracking**
2. **Backtracking with MRV (Minimum Remaining Values) + LCV (Least Constraining Value) heuristics**
3. **Backtracking with MRV + LCV + AC-3 (Arc Consistency)**

**CSP formulation**
- **Variables:** `Q1, Q2, ..., Q8` — `Qi` is the row of the queen placed in column `i`.
- **Domain:** `{1, 2, ..., 8}` for every variable.
- **Constraints:**
  - Row constraint: `Qi != Qj` for all `i != j`
  - Diagonal constraint: `|Qi - Qj| != |i - j|` for all `i != j`

We compare the number of **search steps** (value-assignment attempts) taken by each method to find the first solution.

## 0. Setup: Variables, Domains, Constraint Check, Board Printer

In [1]:
from collections import deque
import copy
import pandas as pd

N = 8
variables = [f"Q{i}" for i in range(1, N + 1)]  # Q1 .. Q8 (columns)

def idx(var):
    """Extract the column index (int) from a variable name like 'Q3' -> 3."""
    return int(var[1:])

def consistent(assignment, var, value):
    """Check row/diagonal constraints of `var=value` against all currently assigned variables."""
    i = idx(var)
    for other_var, other_val in assignment.items():
        j = idx(other_var)
        if other_val == value or abs(other_val - value) == abs(i - j):
            return False
    return True

def print_board(solution):
    """Print an 8x8 board. solution: dict {Qi: row} or list of rows indexed by column."""
    if isinstance(solution, dict):
        rows = [solution[f"Q{c}"] for c in range(1, N + 1)]
    else:
        rows = solution
    for r in range(1, N + 1):
        line = ""
        for c in range(1, N + 1):
            line += "Q " if rows[c - 1] == r else ". "
        print(line)

def to_row_list(solution_dict):
    return [solution_dict[f"Q{c}"] for c in range(1, N + 1)]

## 1. Plain Backtracking

Variables are assigned in the fixed order `Q1, Q2, ..., Q8`, and values are tried in the
fixed order `1, 2, ..., 8`. Every time we attempt to assign a value to a variable, we count
it as one **search step**.

In [2]:
def backtrack_plain(assignment, domains, steps):
    if len(assignment) == N:
        return dict(assignment)

    # Fixed variable order: first unassigned variable Q1..Q8
    var = next(v for v in variables if v not in assignment)

    for value in domains[var]:
        steps[0] += 1
        if consistent(assignment, var, value):
            assignment[var] = value
            result = backtrack_plain(assignment, domains, steps)
            if result:
                return result
            del assignment[var]
    return None

In [3]:
domains_plain = {v: list(range(1, N + 1)) for v in variables}
steps_plain = [0]

solution_plain = backtrack_plain({}, domains_plain, steps_plain)

print("=== Plain Backtracking ===")
print("Solution found:", solution_plain is not None)
print("Solution (row list):", to_row_list(solution_plain))
print("Search steps:", steps_plain[0])
print()
print("Board:")
print_board(solution_plain)

=== Plain Backtracking ===
Solution found: True
Solution (row list): [1, 5, 8, 6, 3, 7, 2, 4]
Search steps: 876

Board:
Q . . . . . . . 
. . . . . . Q . 
. . . . Q . . . 
. . . . . . . Q 
. Q . . . . . . 
. . . Q . . . . 
. . . . . Q . . 
. . Q . . . . . 


## 2. Backtracking with MRV + LCV Heuristics

- **MRV (Minimum Remaining Values):** at each step, pick the unassigned variable with the
  *smallest* current domain (most constrained variable first).
- **LCV (Least Constraining Value):** for the chosen variable, try values in the order that
  rules out the *fewest* options in the domains of the remaining unassigned variables.
- We also use **forward checking**: after assigning a variable, we prune inconsistent values
  from the domains of its unassigned neighbors, and restore them on backtrack. This is what
  allows MRV to see a meaningfully shrunk domain rather than the full 1..8 range.

At every decision point we print which variable MRV selected and the value order LCV produced.

In [4]:
def select_unassigned_var_mrv(assignment, domains):
    """Return the unassigned variable with the smallest domain (ties broken by variable order)."""
    unassigned = [v for v in variables if v not in assignment]
    return min(unassigned, key=lambda var: len(domains[var]))

def order_domain_values_lcv(var, assignment, domains):
    """Order the domain of `var` by increasing number of values it eliminates from neighbors."""
    i = idx(var)

    def count_conflicts(value):
        count = 0
        for other in variables:
            if other == var or other in assignment:
                continue
            j = idx(other)
            for val2 in domains[other]:
                if val2 == value or abs(val2 - value) == abs(i - j):
                    count += 1
        return count

    return sorted(domains[var], key=count_conflicts)

def forward_check(var, value, assignment, domains):
    """
    Prune domains[other] of values inconsistent with var=value, for every unassigned neighbor.
    Returns (pruned, failure):
        pruned  - dict {other_var: [removed values]} so the caller can restore them
        failure - True if some domain became empty (dead end)
    """
    i = idx(var)
    pruned = {}
    failure = False
    for other in variables:
        if other == var or other in assignment:
            continue
        j = idx(other)
        removed = [v for v in domains[other]
                   if v == value or abs(v - value) == abs(i - j)]
        if removed:
            for r in removed:
                domains[other].remove(r)
            pruned[other] = removed
        if not domains[other]:
            failure = True
            break
    return pruned, failure

def restore(pruned, domains):
    for var, vals in pruned.items():
        domains[var].extend(vals)

In [5]:
def backtrack_mrv_lcv(assignment, domains, steps, log=True):
    if len(assignment) == N:
        return dict(assignment)

    var = select_unassigned_var_mrv(assignment, domains)
    if log:
        print(f"MRV -> selects {var}  (current domain size {len(domains[var])}: {domains[var]})")

    values = order_domain_values_lcv(var, assignment, domains)
    if log:
        print(f"LCV -> value order for {var}: {values}")

    for value in values:
        steps[0] += 1
        if consistent(assignment, var, value):
            assignment[var] = value
            pruned, failure = forward_check(var, value, assignment, domains)
            if not failure:
                result = backtrack_mrv_lcv(assignment, domains, steps, log)
                if result:
                    return result
            restore(pruned, domains)
            del assignment[var]
    return None

In [6]:
domains_mrv = {v: list(range(1, N + 1)) for v in variables}
steps_mrv = [0]

print("=== Backtracking + MRV + LCV (decision trace) ===")
solution_mrv = backtrack_mrv_lcv({}, domains_mrv, steps_mrv, log=True)

print()
print("Solution found:", solution_mrv is not None)
print("Solution (row list):", to_row_list(solution_mrv))
print("Search steps:", steps_mrv[0])
print()
print("Board:")
print_board(solution_mrv)

=== Backtracking + MRV + LCV (decision trace) ===
MRV -> selects Q1  (current domain size 8: [1, 2, 3, 4, 5, 6, 7, 8])
LCV -> value order for Q1: [1, 2, 3, 4, 5, 6, 7, 8]
MRV -> selects Q2  (current domain size 6: [3, 4, 5, 6, 7, 8])
LCV -> value order for Q2: [4, 6, 8, 3, 5, 7]
MRV -> selects Q3  (current domain size 4: [2, 6, 7, 8])
LCV -> value order for Q3: [7, 8, 2, 6]
MRV -> selects Q4  (current domain size 2: [3, 5])
LCV -> value order for Q4: [3, 5]
MRV -> selects Q6  (current domain size 1: [2])
LCV -> value order for Q6: [2]
MRV -> selects Q5  (current domain size 2: [6, 8])
LCV -> value order for Q5: [6, 8]
MRV -> selects Q7  (current domain size 1: [5])
LCV -> value order for Q7: [5]
MRV -> selects Q7  (current domain size 1: [5])
LCV -> value order for Q7: [5]
MRV -> selects Q6  (current domain size 1: [2])
LCV -> value order for Q6: [2]
MRV -> selects Q5  (current domain size 1: [8])
LCV -> value order for Q5: [8]
MRV -> selects Q4  (current domain size 2: [3, 5])
LCV -> 

## 3. Backtracking with MRV + LCV + AC-3

**AC-3 (Arc Consistency Algorithm #3)** enforces arc consistency over all constraint arcs
`(Xi, Xj)`. An arc `(Xi, Xj)` is consistent if, for every value in `Xi`'s domain, there is
some value in `Xj`'s domain that satisfies the constraint between them. `revise(Xi, Xj)`
removes values from `Xi`'s domain that have no such support, enabling early failure
detection (a domain becoming empty means the current partial assignment cannot lead to a
solution, so we can prune immediately instead of exploring further).

We use AC-3 in two places:
1. **Preprocessing** — run AC-3 once on the full set of arcs before search starts.
2. **MAC (Maintaining Arc Consistency)** — after every assignment during search, we fix the
   assigned variable's domain to the chosen value and re-run AC-3 restricted to arcs
   `(Xk, var)` for all unassigned `Xk`, propagating the consequences.

Domain reductions discovered by AC-3 are printed as they happen.

In [7]:
def revise(domains, xi, xj):
    """Remove values from domains[xi] that have no supporting value in domains[xj]."""
    i, j = idx(xi), idx(xj)
    revised = False
    for x in domains[xi][:]:
        # Is there some y in domains[xj] consistent with x (different row, different diagonal)?
        if not any(y != x and abs(x - y) != abs(i - j) for y in domains[xj]):
            domains[xi].remove(x)
            revised = True
    return revised

def AC3(domains, arcs=None, log=False):
    """Standard AC-3. If `arcs` is None, initializes the queue with all arcs between all variable pairs."""
    queue = deque(arcs if arcs is not None else
                  [(xi, xj) for xi in variables for xj in variables if xi != xj])
    while queue:
        xi, xj = queue.popleft()
        if revise(domains, xi, xj):
            if log:
                print(f"  AC-3 revise({xi}, {xj}): domain[{xi}] -> {domains[xi]}")
            if not domains[xi]:
                if log:
                    print(f"  AC-3: domain of {xi} became EMPTY -> inconsistent!")
                return False
            for xk in variables:
                if xk != xi and xk != xj:
                    queue.append((xk, xi))
    return True

In [8]:
def backtrack_ac3(assignment, domains, steps, log=True):
    if len(assignment) == N:
        return dict(assignment)

    var = select_unassigned_var_mrv(assignment, domains)
    if log:
        print(f"MRV -> selects {var}  (current domain size {len(domains[var])}: {domains[var]})")

    values = order_domain_values_lcv(var, assignment, domains)
    if log:
        print(f"LCV -> value order for {var}: {values}")

    for value in values:
        steps[0] += 1
        if consistent(assignment, var, value):
            assignment[var] = value
            saved_domains = copy.deepcopy(domains)
            domains[var] = [value]

            arcs = [(xk, var) for xk in variables if xk not in assignment]
            if log:
                print(f"  Running AC-3 after assigning {var}={value} ...")
            if AC3(domains, arcs, log=log):
                result = backtrack_ac3(assignment, domains, steps, log)
                if result:
                    return result

            domains.clear()
            domains.update(saved_domains)
            del assignment[var]
    return None

In [9]:
# Initial AC-3 preprocessing (before search starts)
domains_ac3 = {v: list(range(1, N + 1)) for v in variables}

print("=== Initial AC-3 preprocessing ===")
consistent_after_init = AC3(domains_ac3, log=True)
print("Arc-consistent after preprocessing:", consistent_after_init)
print("Domains after initial AC-3:")
for v in variables:
    print(f"  {v}: {domains_ac3[v]}")

=== Initial AC-3 preprocessing ===
Arc-consistent after preprocessing: True
Domains after initial AC-3:
  Q1: [1, 2, 3, 4, 5, 6, 7, 8]
  Q2: [1, 2, 3, 4, 5, 6, 7, 8]
  Q3: [1, 2, 3, 4, 5, 6, 7, 8]
  Q4: [1, 2, 3, 4, 5, 6, 7, 8]
  Q5: [1, 2, 3, 4, 5, 6, 7, 8]
  Q6: [1, 2, 3, 4, 5, 6, 7, 8]
  Q7: [1, 2, 3, 4, 5, 6, 7, 8]
  Q8: [1, 2, 3, 4, 5, 6, 7, 8]


In [10]:
steps_ac3 = [0]

print("\n=== Backtracking + MRV + LCV + AC-3 (decision trace) ===")
solution_ac3 = backtrack_ac3({}, domains_ac3, steps_ac3, log=True)

print()
print("Solution found:", solution_ac3 is not None)
print("Solution (row list):", to_row_list(solution_ac3))
print("Search steps:", steps_ac3[0])
print()
print("Board:")
print_board(solution_ac3)


=== Backtracking + MRV + LCV + AC-3 (decision trace) ===
MRV -> selects Q1  (current domain size 8: [1, 2, 3, 4, 5, 6, 7, 8])
LCV -> value order for Q1: [1, 2, 3, 4, 5, 6, 7, 8]
  Running AC-3 after assigning Q1=1 ...
  AC-3 revise(Q2, Q1): domain[Q2] -> [3, 4, 5, 6, 7, 8]
  AC-3 revise(Q3, Q1): domain[Q3] -> [2, 4, 5, 6, 7, 8]
  AC-3 revise(Q4, Q1): domain[Q4] -> [2, 3, 5, 6, 7, 8]
  AC-3 revise(Q5, Q1): domain[Q5] -> [2, 3, 4, 6, 7, 8]
  AC-3 revise(Q6, Q1): domain[Q6] -> [2, 3, 4, 5, 7, 8]
  AC-3 revise(Q7, Q1): domain[Q7] -> [2, 3, 4, 5, 6, 8]
  AC-3 revise(Q8, Q1): domain[Q8] -> [2, 3, 4, 5, 6, 7]
MRV -> selects Q2  (current domain size 6: [3, 4, 5, 6, 7, 8])
LCV -> value order for Q2: [4, 6, 8, 3, 5, 7]
  Running AC-3 after assigning Q2=4 ...
  AC-3 revise(Q3, Q2): domain[Q3] -> [2, 6, 7, 8]
  AC-3 revise(Q4, Q2): domain[Q4] -> [3, 5, 7, 8]
  AC-3 revise(Q5, Q2): domain[Q5] -> [2, 3, 6, 8]
  AC-3 revise(Q6, Q2): domain[Q6] -> [2, 3, 5, 7]
  AC-3 revise(Q7, Q2): domain[Q7] -> [2,

## 4. Comparison of the Three Methods

In [11]:
comparison = pd.DataFrame({
    "Method": [
        "Plain Backtracking",
        "Backtracking + MRV + LCV",
        "Backtracking + MRV + LCV + AC-3"
    ],
    "Solution Found": [
        solution_plain is not None,
        solution_mrv is not None,
        solution_ac3 is not None
    ],
    "Search Steps": [
        steps_plain[0],
        steps_mrv[0],
        steps_ac3[0]
    ]
})

comparison

,Method,Solution Found,Search Steps
0,Plain Backtracking,True,876
1,Backtracking + MRV + LCV,True,36
2,Backtracking + MRV + LCV + AC-3,True,14


In [12]:
print(comparison.to_string(index=False))

print()
print("Observation:")
print(f"- Plain backtracking took {steps_plain[0]} steps.")
print(f"- MRV + LCV reduced this to {steps_mrv[0]} steps "
      f"({steps_plain[0] / max(steps_mrv[0], 1):.1f}x fewer).")
print(f"- Adding AC-3 (MAC) further reduced it to {steps_ac3[0]} steps "
      f"({steps_plain[0] / max(steps_ac3[0], 1):.1f}x fewer than plain backtracking).")
print("Heuristics (MRV/LCV) cut down the branching factor by making smarter choices,")
print("while AC-3 detects dead ends earlier by propagating constraints across all")
print("unassigned variables, pruning the search tree before it is even explored.")

                         Method  Solution Found  Search Steps
             Plain Backtracking            True           876
       Backtracking + MRV + LCV            True            36
Backtracking + MRV + LCV + AC-3            True            14

Observation:
- Plain backtracking took 876 steps.
- MRV + LCV reduced this to 36 steps (24.3x fewer).
- Adding AC-3 (MAC) further reduced it to 14 steps (62.6x fewer than plain backtracking).
Heuristics (MRV/LCV) cut down the branching factor by making smarter choices,
while AC-3 detects dead ends earlier by propagating constraints across all
unassigned variables, pruning the search tree before it is even explored.
